# Milestone 2: RAG Exploration

This notebook has the initial explorations of the RAG pipeline.

In [22]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
import faiss
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from groq import Groq
import os
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
# -------------------------
# Load data
# -------------------------
meta = pd.read_json("../data/raw/meta_Toys_and_Games.jsonl", lines=True, nrows=50000)
review = pd.read_json("../data/raw/Toys_and_Games.jsonl", lines=True, nrows=50000)

cleaned_meta = meta.drop(columns=['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'], errors='ignore')
cleaned_meta.head()

reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns=['images', 'timestamp', 'user_id', 'verified_purchase'], errors='ignore')
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [24]:
# -------------------------
# Clean text columns
# -------------------------
cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
).str.lower()

cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
).str.lower()

cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
).str.lower()

cleaned_meta['title'] = cleaned_meta['title'].str.lower()

cleaned_meta = cleaned_meta[
    (cleaned_meta['title'].str.strip() != '') &
    (cleaned_meta['description'].str.strip() != '') &
    (cleaned_meta['features'].str.strip() != '') &
    (cleaned_meta['categories'].str.strip() != '')
].reset_index(drop=True)

In [25]:
# -------------------------
# Prepare review text per product
# -------------------------
cleaned_reviews = cleaned_reviews.copy()
review_text_cols = [col for col in ['title', 'text'] if col in cleaned_reviews.columns]
cleaned_reviews['combined_review_text'] = cleaned_reviews[review_text_cols].fillna('').agg(' '.join, axis=1)
cleaned_reviews['combined_review_text'] = cleaned_reviews['combined_review_text'].str.lower()

grouped_reviews = (
    cleaned_reviews.groupby('parent_asin')['combined_review_text']
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

rag_df = cleaned_meta.merge(grouped_reviews, on='parent_asin', how='left')
rag_df['combined_review_text'] = rag_df['combined_review_text'].fillna('')

In [26]:
# -------------------------
# Build product documents
# -------------------------
products = (
    rag_df['title'] + ' ' +
    rag_df['description'] + ' ' +
    rag_df['features'] + ' ' +
    rag_df['categories'] + ' ' +
    rag_df['combined_review_text']
).tolist()

In [27]:
# -------------------------
# Embeddings and vector store
# -------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6412.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
# -------------------------
# Retriever
# -------------------------

def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return rag_df.iloc[indices[0]]

# -------------------------
# Building context
# -------------------------
# Implemented with the help of chatGPT
def build_context(docs):
    blocks = []
    for _, row in docs.iterrows():
        review_snippet = row.get('combined_review_text', '')[:500]

        block = (
            f"Product ASIN: {row.get('parent_asin', 'N/A')}\n"
            f"Title: {row.get('title', '')}\n"
            f"Description: {row.get('description', '')}\n"
            f"Features: {row.get('features', '')}\n"
            f"Categories: {row.get('categories', '')}\n"
            f"Review Evidence: {review_snippet}\n"
        )
        blocks.append(block)
    return "\n\n".join(blocks)


# -------------------------
# Wrapper function
# -------------------------
def retrieve_and_build_context(query):
    docs = retrieve(query, top_k=5)
    return build_context(docs)

In [29]:
# -------------------------
# Prompt variants
# -------------------------
prompt1 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

prompt2 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Keep the answer shorter than 3 sentences.
- Make sure nothing is repeated, and only necessary details are written.
- If the answer is not in the context, say: "The context does not provide enough information."

Context:
{context}

Input:
{input}

Answer:
"""
)

prompt3 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Be clear and helpful
- Give specific statements instead of general ones. 
- If something is missing, say "not enough context to answer your question"

Context:
{context}

Question:
{input}

Answer:
"""
)

In [30]:
import os
os.environ["GROQ_API_KEY"] = "REMOVED_GROQ_API_KEY"

In [31]:
# -------------------------
# Rag Pipeline
# -------------------------
# Implemented with the help of chatGPT

llm = ChatGroq(model="llama-3.1-8b-instant")

prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))


===== prompt1 =====
The answer is present in the context.

The context suggests that Sorry! board game is not suitable for children under 8 years old due to the potential frustration of having their pawns sent back to the starting line. However, it is recommended for children as young as 6 years old.

A suitable option for kids age 8 and up is mentioned in the context of the Amazon Exclusive description for the Sorry! game, but not explicitly stated for that age group in the description, although it says for ages 6 and up. However, the other context about "children under age 8 to handle" suggests that age 8 might be a suitable age range.

However, a more explicit answer can be found in the context of other products. The product with ASIN B07P2VVFVM is recommended for ages 6 and up, but another option could be "chess checkers & tic tac toe with dual sided gameboard" because it is recommended for ages 6 and up.

===== prompt2 =====
The Sorry! board game is suitable for kids as young as 

In [32]:
rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm
    | StrOutputParser()
)

queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


QUERY: A good board game for kids age 8 and up
The context mentions a suitable age range for the "Sorry!" game as 6 and up. However, it also mentions that children under 8 might have trouble handling frustration from the game, implying that 8 and up might be a more suitable age range.

A more suitable answer is found in the context of the product "Product ASIN: B07P2VVFVM" titled "chess checkers & tic tac toe with dual sided gameboard." The context states "Recommended for ages 6 and up," which is close, but the product titled "Jumbo pick up sticks classic wooden game" also has a similar age recommendation and a different game, however, "Product ASIN: B07P2VVFVM" is the one that matches the requested age range.

The product "Product ASIN: B07P2VVFVM" titled "Chess Checkers & Tic Tac Toe with Dual Sided Gameboard" is the answer.

QUERY: A toy for toddlers
There are several toys for toddlers mentioned in the context:

1. dopomt mini musical toys for toddler, guitar electronic toy piano a

In [39]:
# Please ignore this was ran to answer step 5. 

# import pandas as pd
# from sentence_transformers import SentenceTransformer
# from langchain_groq import ChatGroq
# import faiss
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableLambda, RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser
# from langchain_community.retrievers import BM25Retriever
# from langchain_core.documents import Document



# cleaned_meta = meta.drop(columns=['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'], errors='ignore')

# reviews = review[review['verified_purchase'] == True]
# cleaned_reviews = reviews.drop(columns=['images', 'timestamp', 'user_id', 'verified_purchase'], errors='ignore')


# # Clean text columns
# cleaned_meta['description'] = cleaned_meta['description'].apply(
#     lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
# ).str.lower()

# cleaned_meta['details'] = cleaned_meta['details'].apply(
#     lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
# ).str.lower()

# cleaned_meta['features'] = cleaned_meta['features'].apply(
#     lambda x: " ".join(x) if isinstance(x, list) else ""
# ).str.lower()

# cleaned_meta['categories'] = cleaned_meta['categories'].apply(
#     lambda x: " ".join(x) if isinstance(x, list) else ""
# ).str.lower()

# cleaned_meta['title'] = cleaned_meta['title'].str.lower()

# cleaned_meta = cleaned_meta[
#     (cleaned_meta['title'].str.strip() != '') &
#     (cleaned_meta['description'].str.strip() != '') &
#     (cleaned_meta['features'].str.strip() != '') &
#     (cleaned_meta['categories'].str.strip() != '')
# ].reset_index(drop=True)


# # Prepare review text per product
# cleaned_reviews = cleaned_reviews.copy()
# review_text_cols = [col for col in ['title', 'text'] if col in cleaned_reviews.columns]
# cleaned_reviews['combined_review_text'] = cleaned_reviews[review_text_cols].fillna('').agg(' '.join, axis=1)
# cleaned_reviews['combined_review_text'] = cleaned_reviews['combined_review_text'].str.lower()

# grouped_reviews = (
#     cleaned_reviews.groupby('parent_asin')['combined_review_text']
#     .apply(lambda x: " ".join(x.astype(str)))
#     .reset_index()
# )

# rag_df = cleaned_meta.merge(grouped_reviews, on='parent_asin', how='left')
# rag_df['combined_review_text'] = rag_df['combined_review_text'].fillna('')


# # Build product text strings
# products = (
#     rag_df['title'] + ' ' +
#     rag_df['description'] + ' ' +
#     rag_df['features'] + ' ' +
#     rag_df['categories'] + ' ' +
#     rag_df['combined_review_text']
# ).tolist()


# # Embeddings and FAISS vector store
# embed_model = SentenceTransformer("all-MiniLM-L6-v2")
# product_embeddings = embed_model.encode(products).astype("float32")

# index = faiss.IndexFlatL2(product_embeddings.shape[1])
# index.add(product_embeddings)


# # Build LangChain Documents for BM25
# # (each Document carries the row index in metadata so we can look up rag_df)
# lc_docs = [
#     Document(page_content=text, metadata={"row_index": i})
#     for i, text in enumerate(products)
# ]


# # BM25 Retriever
# # LangChain's BM25Retriever handles tokenisation internally
# bm25_retriever = BM25Retriever.from_documents(lc_docs, k=5)


# # Semantic Retriever (FAISS-based, returns LangChain Documents)
# def semantic_retrieve(query: str, top_k: int = 5) -> list[Document]:
#     query_embedding = embed_model.encode([query]).astype("float32")
#     distances, indices = index.search(query_embedding, top_k)
#     results = []
#     for idx in indices[0]:
#         results.append(
#             Document(
#                 page_content=products[idx],
#                 metadata={"row_index": int(idx)}
#             )
#         )
#     return results


# # Hybrid Retriever: merge BM25 + semantic, deduplicate by row_index
# def hybrid_retriever(query: str, top_k: int = 5) -> list[Document]:
#     bm25_results = bm25_retriever.invoke(query)
#     semantic_results = semantic_retrieve(query, top_k=top_k)

#     seen_indices = set()
#     merged = []

#     # Interleave results from both retrievers to preserve ranking signal
#     for bm25_doc, sem_doc in zip(bm25_results, semantic_results):
#         for doc in (bm25_doc, sem_doc):
#             row_idx = doc.metadata.get("row_index")
#             if row_idx not in seen_indices:
#                 seen_indices.add(row_idx)
#                 merged.append(doc)

#     # Handle any leftover docs if one list is longer than the other
#     for doc in bm25_results + semantic_results:
#         row_idx = doc.metadata.get("row_index")
#         if row_idx not in seen_indices:
#             seen_indices.add(row_idx)
#             merged.append(doc)

#     return merged[:top_k]


# # Context builder (accepts list[Document])
# def build_context(docs: list[Document]) -> str:
#     blocks = []
#     for doc in docs:
#         row_idx = doc.metadata.get("row_index")
#         if row_idx is None:
#             continue
#         row = rag_df.iloc[row_idx]
#         review_snippet = row.get('combined_review_text', '')[:500]
#         block = (
#             f"Product ASIN: {row.get('parent_asin', 'N/A')}\n"
#             f"Title: {row.get('title', '')}\n"
#             f"Description: {row.get('description', '')}\n"
#             f"Features: {row.get('features', '')}\n"
#             f"Categories: {row.get('categories', '')}\n"
#             f"Review Evidence: {review_snippet}\n"
#         )
#         blocks.append(block)
#     return "\n\n".join(blocks)


# # Prompt variants
# prompt1 = ChatPromptTemplate.from_template(
# """
# You must answer using ONLY the information in the context.

# - If the answer is present, extract and summarize it clearly.
# - Do NOT say "I don't know" if the answer exists in the context.
# - Only say "I don't know" if the context truly does not contain the answer.

# Context:
# {context}

# Question:
# {question}

# Answer:
# """
# )

# prompt2 = ChatPromptTemplate.from_template(
# """
# You must answer using ONLY the information in the context.

# - Keep the answer shorter than 3 sentences.
# - Make sure nothing is repeated, and only necessary details are written.
# - If the answer is not in the context, say: "The context does not provide enough information."

# Context:
# {context}

# Input:
# {question}

# Answer:
# """
# )

# prompt3 = ChatPromptTemplate.from_template(
# """
# You must answer using ONLY the information in the context.

# - Be clear and helpful
# - Give specific statements instead of general ones.
# - If something is missing, say "not enough context to answer your question"

# Context:
# {context}

# Question:
# {question}

# Answer:
# """
# )


# # Hybrid RAG Pipeline
# llm = ChatGroq(model="llama-3.1-8b-instant")

# rag_chain = (
#     {
#         "context": RunnableLambda(hybrid_retriever) | RunnableLambda(build_context),
#         "question": RunnablePassthrough()
#     }
#     | prompt1
#     | llm
#     | StrOutputParser()
# )


# # Run example queries
# if __name__ == "__main__":
#     queries = [
#         "A good board game for kids age 8 and up",
#         "A toy for toddlers",
#         "Educational toys for kids"
#         "A good gift for children who like biilding toys"
#         "A fun indoor activity toy for kids"
#     ]

#     for q in queries:
#         print(f"\nQUERY: {q}")
#         print(rag_chain.invoke(q))

#     # Prompt comparison on a single query
#     prompts = {
#         "prompt1": prompt1,
#         "prompt2": prompt2,
#         "prompt3": prompt3,
#     }

#     query = "A good board game for kids age 8 and up"

#     for name, prompt in prompts.items():
#         test_chain = (
#             {
#                 "context": RunnableLambda(hybrid_retriever) | RunnableLambda(build_context),
#                 "question": RunnablePassthrough()
#             }
#             | prompt
#             | llm
#             | StrOutputParser()
#         )
#         print(f"\n===== {name} =====")
#         print(test_chain.invoke(query))

In [ ]:
# queries = [
#     "A good board game for kids age 8 and up",
#     "A toy for toddlers",
#     "Educational toys for kids",
#     "A good gift for a child who likes building toys",
#     "A fun indoor activity toy for kids"
# ]

# for q in queries:
#     docs = hybrid_retriever(q, top_k=5)
#     context = build_context(docs)
#     answer = rag_chain.invoke(q)

#     print("\n" + "="*80)
#     print("QUERY:", q)
#     print("\nANSWER:")
#     print(answer)
#     print("\nCONTEXT:")
#     print(context)


QUERY: A good board game for kids age 8 and up

ANSWER:
Based on the context, a good board game for kids age 8 and up is the "Sorry!" board game (Product ASIN: B00000IWD0). It is described as a classic game that is fun for adults and older siblings too, and it is suitable for children as young as 6 years old. It is also mentioned that children under 8 may find the game frustrating, but it can help them develop strategic thinking and plot their own revenge.

CONTEXT:
Product ASIN: B09VSRT17N
Title: 30" large dart board for kids-best festival gifts for boys girls
Description: dinosaur toys for 3-12 year old boys,30”large dart board kids toys age 4-12,indoor outdoor games birthday christmas stocking stuffers gifts for 3-12 year old boys toys unicorn toys for 3-12 year old girls,25”large dart board kids toys for 6-12 year old girls teens party outdoor games,christmas birthday gifts for girls age 3-12
Features: 【30 inch huge dinosaur toys】all kids love dinosaurs!this dinosaur dart borad 30